In [ ]:
# Exploratory Data Analysis (EDA) — WHO Life Expectancy Dataset

**Dataset:** Life Expectancy Data from the Global Health Observatory (GHO) / WHO  
**Period:** 2000–2015 | **Countries:** 193  
**Goal:** Systematically explore the data, understand distributions, identify missing values, detect outliers, and uncover relationships between health, economic, and social factors and life expectancy.

## 1. Import Libraries

In [ ]:
# Install dependencies (uncomment if running on Google Colab)
# !pip install pandas numpy matplotlib seaborn scipy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')

# Plotting settings
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print("Libraries imported successfully.")

## 2. Load the Dataset

In [ ]:
# Load the dataset
# For Google Colab, upload the file or mount Google Drive and update the path accordingly
# from google.colab import files
# uploaded = files.upload()

df = pd.read_csv('Life_Expectancy_Data.csv')

# Clean column names: strip leading/trailing whitespace
df.columns = df.columns.str.strip()

print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head(10)

## 3. Data Overview & Structure

In [ ]:
# Data types and non-null counts
df.info()

In [ ]:
# Descriptive statistics for numerical features
df.describe().T

In [ ]:
# Categorical feature overview
print("Unique countries:", df['Country'].nunique())
print("Year range:", df['Year'].min(), "–", df['Year'].max())
print("\nStatus distribution:")
print(df['Status'].value_counts())
print("\nSample countries:")
print(sorted(df['Country'].unique())[:20])

## 4. Missing Values Analysis

In [ ]:
# Missing values count and percentage
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2)
})
missing = missing[missing['Missing Count'] > 0].sort_values('Missing %', ascending=False)
print(f"Columns with missing values: {len(missing)} out of {df.shape[1]}\n")
missing

In [ ]:
# Visualize missing values
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart of missing percentages
missing_sorted = missing.sort_values('Missing %', ascending=True)
axes[0].barh(missing_sorted.index, missing_sorted['Missing %'], color='coral')
axes[0].set_xlabel('Missing (%)')
axes[0].set_title('Missing Values by Feature')

# Heatmap of missing values
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='viridis', ax=axes[1])
axes[1].set_title('Missing Value Heatmap (yellow = missing)')

plt.tight_layout()
plt.show()

## 5. Distribution of Target Variable — Life Expectancy

In [ ]:
# Distribution of Life Expectancy
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogram with KDE
sns.histplot(df['Life expectancy'].dropna(), kde=True, bins=30, color='steelblue', ax=axes[0])
axes[0].set_title('Distribution of Life Expectancy')
axes[0].axvline(df['Life expectancy'].mean(), color='red', linestyle='--', label=f"Mean: {df['Life expectancy'].mean():.1f}")
axes[0].axvline(df['Life expectancy'].median(), color='green', linestyle='--', label=f"Median: {df['Life expectancy'].median():.1f}")
axes[0].legend()

# Box plot
sns.boxplot(y=df['Life expectancy'], color='steelblue', ax=axes[1])
axes[1].set_title('Box Plot of Life Expectancy')

# Box plot by Status
sns.boxplot(x='Status', y='Life expectancy', data=df, palette='Set2', ax=axes[2])
axes[2].set_title('Life Expectancy: Developed vs Developing')

plt.tight_layout()
plt.show()

# Skewness and Kurtosis
print(f"Skewness: {df['Life expectancy'].skew():.4f}")
print(f"Kurtosis: {df['Life expectancy'].kurtosis():.4f}")

## 6. Distribution of Numerical Features

In [ ]:
# Histograms for all numerical features
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols.remove('Year')  # Year is categorical in nature

n_cols = 4
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.histplot(df[col].dropna(), kde=True, bins=30, ax=axes[i], color='steelblue')
    axes[i].set_title(col, fontsize=11)
    axes[i].set_xlabel('')

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribution of Numerical Features', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 7. Outlier Detection (Box Plots)

In [ ]:
# Box plots for all numerical features to detect outliers
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.boxplot(y=df[col], ax=axes[i], color='lightcoral')
    axes[i].set_title(col, fontsize=11)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Box Plots — Outlier Detection', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Quantify outliers using IQR method
print("Outlier counts (IQR method):\n")
outlier_summary = {}
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    if n_outliers > 0:
        outlier_summary[col] = n_outliers

outlier_df = pd.DataFrame.from_dict(outlier_summary, orient='index', columns=['Outlier Count'])
outlier_df = outlier_df.sort_values('Outlier Count', ascending=False)
outlier_df['Outlier %'] = (outlier_df['Outlier Count'] / len(df) * 100).round(2)
outlier_df

## 8. Correlation Analysis

In [ ]:
# Correlation heatmap
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(16, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, vmin=-1, vmax=1)
plt.title('Correlation Heatmap of Numerical Features', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Top correlations with Life expectancy
life_corr = corr_matrix['Life expectancy'].drop('Life expectancy').sort_values(ascending=False)
print("Top correlations with Life Expectancy:\n")

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['forestgreen' if v > 0 else 'crimson' for v in life_corr.values]
life_corr.plot(kind='barh', color=colors, ax=ax)
ax.set_xlabel('Correlation Coefficient')
ax.set_title('Feature Correlations with Life Expectancy')
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

print(life_corr.to_string())

## 9. Scatter Plots — Key Relationships with Life Expectancy

In [ ]:
# Scatter plots of top correlated features vs Life expectancy
top_features = ['Schooling', 'Income composition of resources', 'BMI',
                'Polio', 'Diphtheria', 'Adult Mortality', 'HIV/AIDS',
                'thinness  1-19 years', 'thinness 5-9 years']

fig, axes = plt.subplots(3, 3, figsize=(18, 15))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    sns.scatterplot(x=df[feat], y=df['Life expectancy'], hue=df['Status'],
                    alpha=0.5, ax=axes[i], palette='Set1', s=20)
    axes[i].set_title(f'Life Expectancy vs {feat}', fontsize=11)
    axes[i].legend(fontsize=8)

plt.suptitle('Key Relationships with Life Expectancy', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 10. Life Expectancy Trends Over Time

In [ ]:
# Average life expectancy over time by Status
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Overall trend
yearly_avg = df.groupby('Year')['Life expectancy'].mean()
axes[0].plot(yearly_avg.index, yearly_avg.values, marker='o', color='steelblue', linewidth=2)
axes[0].set_title('Average Life Expectancy Over Time (All Countries)')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Life Expectancy')
axes[0].grid(True, alpha=0.3)

# By development status
for status, group in df.groupby('Status'):
    yearly = group.groupby('Year')['Life expectancy'].mean()
    axes[1].plot(yearly.index, yearly.values, marker='o', linewidth=2, label=status)
axes[1].set_title('Average Life Expectancy: Developed vs Developing')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Life Expectancy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Top & Bottom Countries by Life Expectancy

In [ ]:
# Top 15 and Bottom 15 countries by average life expectancy
country_avg = df.groupby('Country')['Life expectancy'].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Top 15
top15 = country_avg.head(15)
top15.sort_values().plot(kind='barh', color='seagreen', ax=axes[0])
axes[0].set_title('Top 15 Countries by Avg Life Expectancy')
axes[0].set_xlabel('Life Expectancy (years)')

# Bottom 15
bottom15 = country_avg.tail(15)
bottom15.sort_values().plot(kind='barh', color='indianred', ax=axes[1])
axes[1].set_title('Bottom 15 Countries by Avg Life Expectancy')
axes[1].set_xlabel('Life Expectancy (years)')

plt.tight_layout()
plt.show()

## 12. Developed vs Developing — Feature Comparison

In [ ]:
# Compare key features between Developed and Developing countries
compare_features = ['Life expectancy', 'Adult Mortality', 'infant deaths', 'Alcohol',
                    'GDP', 'BMI', 'Schooling', 'HIV/AIDS', 'Income composition of resources']

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, feat in enumerate(compare_features):
    sns.boxplot(x='Status', y=feat, data=df, palette='Set2', ax=axes[i])
    axes[i].set_title(feat, fontsize=11)

plt.suptitle('Feature Comparison: Developed vs Developing', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 13. Pair Plot of Key Features

In [ ]:
# Pair plot of selected key features
pair_features = ['Life expectancy', 'Schooling', 'Income composition of resources',
                 'Adult Mortality', 'GDP', 'Status']

sns.pairplot(df[pair_features].dropna(), hue='Status', palette='Set1',
             diag_kind='kde', plot_kws={'alpha': 0.4, 's': 15})
plt.suptitle('Pair Plot of Key Features', y=1.02, fontsize=16)
plt.show()

## 14. Immunization Coverage Analysis

In [ ]:
# Immunization coverage over time: Hepatitis B, Polio, Diphtheria
immunization_cols = ['Hepatitis B', 'Polio', 'Diphtheria']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(immunization_cols):
    for status, group in df.groupby('Status'):
        yearly = group.groupby('Year')[col].mean()
        axes[i].plot(yearly.index, yearly.values, marker='o', linewidth=2, label=status)
    axes[i].set_title(f'{col} Coverage Over Time')
    axes[i].set_xlabel('Year')
    axes[i].set_ylabel('Coverage (%)')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 15. GDP & Economic Factors vs Life Expectancy

In [ ]:
# GDP vs Life Expectancy (log scale for GDP)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# GDP (log scale)
df_gdp = df[df['GDP'] > 0].copy()
sns.scatterplot(x=np.log10(df_gdp['GDP']), y=df_gdp['Life expectancy'],
                hue=df_gdp['Status'], alpha=0.5, palette='Set1', ax=axes[0], s=20)
axes[0].set_xlabel('log10(GDP)')
axes[0].set_title('Life Expectancy vs GDP (log scale)')
axes[0].legend()

# Total expenditure on health
sns.scatterplot(x='Total expenditure', y='Life expectancy', hue='Status',
                data=df, alpha=0.5, palette='Set1', ax=axes[1], s=20)
axes[1].set_title('Life Expectancy vs Total Health Expenditure')
axes[1].legend()

plt.tight_layout()
plt.show()

## 16. Mortality & Disease Factors

In [ ]:
# Mortality and disease factors
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.scatterplot(x='Adult Mortality', y='Life expectancy', hue='Status',
                data=df, alpha=0.5, palette='Set1', ax=axes[0, 0], s=20)
axes[0, 0].set_title('Life Expectancy vs Adult Mortality')

sns.scatterplot(x='infant deaths', y='Life expectancy', hue='Status',
                data=df, alpha=0.5, palette='Set1', ax=axes[0, 1], s=20)
axes[0, 1].set_title('Life Expectancy vs Infant Deaths')

sns.scatterplot(x='HIV/AIDS', y='Life expectancy', hue='Status',
                data=df, alpha=0.5, palette='Set1', ax=axes[1, 0], s=20)
axes[1, 0].set_title('Life Expectancy vs HIV/AIDS Deaths (0-4 years)')

sns.scatterplot(x='Measles', y='Life expectancy', hue='Status',
                data=df, alpha=0.5, palette='Set1', ax=axes[1, 1], s=20)
axes[1, 1].set_title('Life Expectancy vs Measles Cases')

for ax in axes.flatten():
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 17. Feature Skewness Analysis

In [ ]:
# Skewness of all numerical features
skewness = df[numeric_cols].skew().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['crimson' if abs(v) > 1 else 'orange' if abs(v) > 0.5 else 'green' for v in skewness.values]
skewness.plot(kind='barh', color=colors, ax=ax)
ax.axvline(x=0, color='black', linewidth=0.5)
ax.axvline(x=-1, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=1, color='gray', linestyle='--', alpha=0.5)
ax.set_title('Feature Skewness (|skew| > 1 = highly skewed)')
ax.set_xlabel('Skewness')
plt.tight_layout()
plt.show()

print("\nHighly skewed features (|skew| > 1):")
print(skewness[abs(skewness) > 1].to_string())

## 18. Summary of EDA Findings

**Key observations from the exploratory analysis:**

1. **Dataset:** 2,938 records across 193 countries (2000–2015), with 22 features including health, economic, and social indicators.

2. **Missing Values:** Several columns have missing data — notably Population, Hepatitis B, and GDP. These will need to be handled during preprocessing.

3. **Target Distribution:** Life expectancy is slightly left-skewed, with a mean around 69 years. Developed countries cluster between 75–85 years, while developing countries show greater variance (40–80 years).

4. **Strongest Positive Correlations with Life Expectancy:**
   - Schooling, Income composition of resources, BMI, Polio/Diphtheria coverage

5. **Strongest Negative Correlations with Life Expectancy:**
   - Adult Mortality, HIV/AIDS, thinness measures

6. **Developed vs Developing Gap:** A consistent ~10-year gap exists between developed and developing nations, though developing countries have shown steady improvement over the period.

7. **Outliers:** Significant outliers exist in features like Population, Measles, infant deaths, GDP, and percentage expenditure — mostly due to high-population countries.

8. **Highly Skewed Features:** Measles, infant deaths, under-five deaths, Population, GDP, and percentage expenditure are heavily right-skewed and may benefit from log transformation in modeling.

9. **Immunization:** Polio and Diphtheria coverage have improved globally, with developing countries closing the gap over time. Hepatitis B coverage remains lower.

10. **GDP Effect:** A clear logarithmic relationship exists — GDP impact on life expectancy plateaus at higher income levels.